# File Validation + AI Verification Reference

This notebook summarizes the file validation and AI-assisted verification flow used in the backend.
It focuses on: extension and MIME checks, size limits, hash computation, text extraction, and Groq verification.

## Dependencies

Core validation: `filetype`
Text extraction: `pypdf`
OCR: `pytesseract` + `Pillow` + Tesseract binary
Groq client: `groq`

In [ ]:
import hashlib
import os
from dataclasses import dataclass
from typing import Dict, Set, Tuple

import filetype


@dataclass(frozen=True)
class DocumentType:
    GOV_ID: str = "GOV_ID"
    RESIDENCY: str = "RESIDENCY"
    RESUME: str = "RESUME"


ALLOWED_TYPES: Dict[str, Dict[str, Set[str]]] = {
    DocumentType.GOV_ID: {
        "extensions": {"pdf", "jpg", "png"},
        "mimes": {"application/pdf", "image/jpeg", "image/png"},
    },
    DocumentType.RESIDENCY: {
        "extensions": {"pdf", "jpg", "png"},
        "mimes": {"application/pdf", "image/jpeg", "image/png"},
    },
    DocumentType.RESUME: {
        "extensions": {"pdf"},
        "mimes": {"application/pdf"},
    },
}

ARCHIVE_EXTENSIONS = {"zip", "rar", "7z", "tar", "gz", "bz2", "xz"}

In [ ]:
def normalize_extension(extension: str) -> str:
    normalized = extension.lower().lstrip(".")
    if normalized == "jpeg":
        return "jpg"
    return normalized


def sniff_file_type(path: str) -> Tuple[str, str]:
    kind = filetype.guess(path)
    if not kind:
        raise ValueError("Unsupported or unrecognized file type.")
    extension = normalize_extension(kind.extension or "")
    mime = (kind.mime or "").lower()
    return extension, mime


def compute_sha256(path: str) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

In [ ]:
def validate_document_file(document_type: str, path: str, max_bytes: int = 10 * 1024 * 1024) -> Tuple[str, str]:
    if document_type not in ALLOWED_TYPES:
        raise ValueError("Invalid document type.")
    if not os.path.exists(path):
        raise ValueError("File path does not exist.")

    size = os.path.getsize(path)
    if size > max_bytes:
        max_mb = max_bytes // (1024 * 1024)
        raise ValueError(f"File exceeds max size of {max_mb} MB.")

    original_extension = normalize_extension(os.path.splitext(path)[1])
    if not original_extension:
        raise ValueError("File must include an extension.")

    sniffed_extension, sniffed_mime = sniff_file_type(path)

    if original_extension in ARCHIVE_EXTENSIONS or sniffed_extension in ARCHIVE_EXTENSIONS:
        raise ValueError("Archive files are not allowed.")

    allowed = ALLOWED_TYPES[document_type]
    if original_extension not in allowed["extensions"]:
        raise ValueError("File extension not allowed for this document type.")
    if sniffed_extension not in allowed["extensions"] or sniffed_mime not in allowed["mimes"]:
        raise ValueError("File content type does not match allowed types.")
    if original_extension != sniffed_extension:
        raise ValueError("File extension does not match file content.")

    return sniffed_extension, sniffed_mime

In [ ]:
sample_path = "/path/to/your/document.pdf"
document_type = DocumentType.GOV_ID

if os.path.exists(sample_path):
    extension, mime = validate_document_file(document_type, sample_path)
    checksum = compute_sha256(sample_path)
    print({"extension": extension, "mime": mime, "checksum": checksum})
else:
    print("Set sample_path to a real file to run the validation.")

## Example: Resume validator

This demo validates a resume file (PDF only) and prints the detected MIME type and SHA-256 hash.

In [ ]:
resume_path = "/path/to/your/resume.pdf"
document_type = DocumentType.RESUME

try:
    if not os.path.exists(resume_path):
        raise FileNotFoundError("Set resume_path to a real PDF file.")

    extension, mime = validate_document_file(document_type, resume_path)
    checksum = compute_sha256(resume_path)
    print({"extension": extension, "mime": mime, "checksum": checksum})
except Exception as exc:
    print(f"Resume validation failed: {exc}")

## Text Extraction (PDF + OCR)

The backend first tries PDF text extraction and falls back to OCR when the text is too short.
OCR is always used for image files.

In [ ]:
def extract_pdf_text(path: str, max_pages: int = 3) -> Tuple[str, int]:
    from pypdf import PdfReader

    reader = PdfReader(path, strict=False)
    page_count = len(reader.pages)
    chunks = []
    for index in range(min(max_pages, page_count)):
        chunks.append(reader.pages[index].extract_text() or "")
    return "\n".join(chunks).strip(), page_count


def ocr_image_text(path: str, lang: str = "eng") -> str:
    from PIL import Image
    import pytesseract

    with Image.open(path) as image:
        image = image.convert("RGB")
        return pytesseract.image_to_string(image, lang=lang).strip()


def ocr_pdf_text(path: str, max_pages: int = 3, lang: str = "eng") -> str:
    import fitz
    from PIL import Image
    import pytesseract

    chunks = []
    with fitz.open(path) as pdf:
        for index, page in enumerate(pdf):
            if index >= max_pages:
                break
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
            mode = "RGB" if pix.alpha == 0 else "RGBA"
            image = Image.frombytes(mode, (pix.width, pix.height), pix.samples)
            if mode == "RGBA":
                image = image.convert("RGB")
            chunks.append(pytesseract.image_to_string(image, lang=lang))
    return "\n".join(chunks).strip()

## Groq Verification Prompt

This mirrors the backend approach: lenient checks that only reject obvious junk.
If text is too short or unclear, it should still pass but recommend manual review.

In [ ]:
import json
from typing import Dict


def build_groq_payload(document_type: str, text: str, metadata: Dict[str, str]) -> Dict[str, str]:
    system_prompt = (
        "You verify if an uploaded document matches the expected type. "
        "Be lenient and only reject if the content is clearly unrelated, random, or junk. "
        "If the text is unclear or too short, return verdict 'clean' and explain that manual review is advised. "
        "Respond ONLY with a JSON object: {\"verdict\": \"clean\"|\"reject\", \"reason\": \"short reason\"}."
    )

    metadata_lines = "\n".join(f"- {key}: {value}" for key, value in sorted(metadata.items()))
    user_prompt = (
        f"Document type: {document_type}\n"
        f"Metadata:\n{metadata_lines}\n\n"
        "Extracted text:\n"
        f"{text}\n"
    )

    return {
        "system": system_prompt,
        "user": user_prompt,
    }


def parse_groq_response(content: str) -> Dict[str, str]:
    start = content.find("{")
    end = content.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("Groq response was not valid JSON.")
    return json.loads(content[start : end + 1])